Modelo de Rede Neural

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

In [ ]:
# importação os arquivos
df_train = pd.read_csv("./dados_projeto/treino.csv")
df_test1 = pd.read_csv("./dados_projeto/teste1.csv")
df_test2 = pd.read_csv("./dados_projeto/teste2.csv")

In [ ]:
# verifica a conexão com a placa de vídeo
gpus = tf.config.list_physical_devices('GPU')
if gpus:
  try:
    for gpu in gpus:
      tf.config.experimental.set_memory_growth(gpu, True)
    print(f"Sucesso! GPU(s) detectada(s): {gpus}")
  except RuntimeError as e:
    print(f"Erro ao configurar a GPU: {e}")
else:
  print("Nenhuma GPU detectada. Rodando na CPU. Verifique a instalação do CUDA/cuDNN.")

In [ ]:
# tipos de variáveis
categorical_onehot_encoder_cols = ['AGRAVTABAC', 'AGRAVDROGA', 'AGRAVAIDS', 'AGRAVDIABE', 'HIV','POP_RUA', 'POP_LIBER', 'POP_IMIG', 'CS_SEXO', 'BACILOSC_E', 'RAIOX_TORA', 'CS_RACA', 'TRATAMENTO']
categorical_target_encoder_cols = ['SG_UF_NOT']
ordinal_cols = ['CS_ESCOL_N', 'CAT_IDADE']
numerical_cols = ['IDADE_ANOS']
target_col = ['ltfu']
feature_cols = categorical_onehot_encoder_cols + categorical_target_encoder_cols + ordinal_cols + numerical_cols

In [ ]:
# mapeia para maiúsculo as features em minúsculo
from sklearn.preprocessing import FunctionTransformer

def map_to_uppercase(df):
  if 'idade_anos' in df.columns:
    df['IDADE_ANOS'] = df['idade_anos']
  return df

age_formatter = FunctionTransformer(map_to_uppercase)

In [ ]:
# categoriza a idade da pessoa
def categorize_age(age):
  if age <= 29:
    return 'jovem_adulto'
  elif age <= 49:
    return 'adulto_meia_idade'
  elif age <= 64:
    return 'adulto_transicao_para_idoso'
  elif age >= 65:
    return 'idoso'
  else:
    return 'ignorado'

def apply_age_categorization(df):
  df['CAT_IDADE'] = df['IDADE_ANOS'].apply(categorize_age)
  return df

age_categorizer = FunctionTransformer(apply_age_categorization)

In [ ]:
# seleciona as colunas desejadas
def filter_cols(df):
  return df[feature_cols]

column_selector = FunctionTransformer(filter_cols)

In [ ]:
# mapeia os dados do dataframe
categorical_onehot_encoder_mapping = {
  'CS_RACA': {
    1: 'branca',
    2: 'preta',
    3: 'amarela',
    4: 'parda',
    5: 'indigena',
    9: 'ignorado',
    0: None
  },
  'AGRAVTABAC': {
    1: 'sim',
    2: 'nao',
    9: 'ignorado',
    0: None
  },
  'AGRAVDROGA': {
    1: 'sim',
    2: 'nao',
    9: 'ignorado',
    0: None
  },
  'AGRAVAIDS': {
    1: 'sim',
    2: 'nao',
    9: 'ignorado',
    0: None
  },
  'AGRAVDIABE': {
    1: 'sim',
    2: 'nao',
    9: 'ignorado',
    0: None
  },
  'HIV': {
    1: 'positivo',
    2: 'negativo',
    3: 'em_andamento',
    4: 'realizado',
    0: None
  },
  'POP_RUA': {
    1: 'sim',
    2: 'nao',
    0: None
  },
  'POP_LIBER': {
    1: 'sim',
    2: 'nao',
    0: None
  },
  'POP_IMIG': {
    1: 'sim',
    2: 'nao',
    0: None
  },
  'BACILOSC_E': {
    1: 'positiva',
    2: 'negativa',
    3: 'nao_realizada',
    4: 'nao_se_aplica',
    0: None
  },
  'RAIOX_TORA': {
    1: 'suspeito',
    2: 'normal',
    3: 'outra_patologia',
    4: 'nao_realizado',
    0: None
  },
  'TRATAMENTO': {
    1: 'caso_novo',
    2: 'recidiva',
    3: 'reingresso_apos_abandono',
    4: 'nao_sabe',
    5: 'transferencia',
    0: None
  }
}
categorical_target_encoder_mapping = {
  'SG_UF_NOT': {
    11: 'ro', 
    12: 'ac', 
    13: 'am', 
    14: 'rr', 
    15: 'pa', 
    16: 'ap', 
    17: 'to',
    21: 'ma', 
    22: 'pi', 
    23: 'ce', 
    24: 'rn', 
    25: 'pb', 
    26: 'pe', 
    27: 'al', 
    28: 'se', 
    29: 'ba',
    31: 'mg', 
    32: 'es', 
    33: 'rj', 
    35: 'sp',
    41: 'pr', 
    42: 'sc', 
    43: 'rs',
    50: 'ms', 
    51: 'mt', 
    52: 'go', 
    53: 'df'
  }
}
ordinal_mapping = {
  'CS_ESCOL_N': {
    0: 'analfabeto',
    1: '1a_a_4a_serie_incompleta_do_ef',
    2: '4a_serie_completa_do_ef',
    3: '5a_a_8a_serie_incompleta_do_ef',
    4: 'ensino_fundamental_completo',
    5: 'ensino_medio_incompleto',
    6: 'ensino_medio_completo',
    7: 'educacao_superior_incompleta',
    8: 'educacao_superior_completa',
    9: 'ignorado',
    10: 'nao_se_aplica'
  }
}

def map_df_data(df, cols_list, mapping_dict):
  for col in cols_list:
    if col in mapping_dict.keys():
      df[col] = df[col].map(mapping_dict[col])
  return df

def map_to_lowercase(df, col):
  df[col] = df[col].str.lower()
  return df

def apply_df_mapping(df):
  df_mapped = map_df_data(df, categorical_onehot_encoder_cols, categorical_onehot_encoder_mapping)
  df_mapped = map_df_data(df_mapped, categorical_target_encoder_cols, categorical_target_encoder_mapping)
  df_mapped = map_df_data(df_mapped, ordinal_cols, ordinal_mapping)
  df_mapped = map_to_lowercase(df_mapped, 'CS_SEXO')
  return df_mapped 

column_mapper = FunctionTransformer(apply_df_mapping)

In [ ]:
# separação das features e do alvo
X_train = df_train.drop(columns=target_col)
y_train = df_train[target_col[0]]

In [ ]:
# imputação dos missings
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from imblearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, TargetEncoder, PowerTransformer

numerical_prep = Pipeline([
  ('imputer', IterativeImputer(random_state=32)),
  ('scaler', PowerTransformer())
])

categorical_onehot_encoder_prep = Pipeline([
  ('imputer', SimpleImputer(strategy='most_frequent')),
  ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

categorical_target_encoder_prep = Pipeline([
  ('imputer', SimpleImputer(strategy='most_frequent')),
  ('encoder', TargetEncoder(cv=5))
])

ordinal_prep = Pipeline([
  ('imputer', SimpleImputer(strategy='most_frequent')),
  ('encoder', TargetEncoder(cv=5))
])

preprocessor = ColumnTransformer([
  ('num', numerical_prep, numerical_cols),
  ('cat_onehot_encoder', categorical_onehot_encoder_prep, categorical_onehot_encoder_cols),
  ('cat_target_encoder', categorical_target_encoder_prep, categorical_target_encoder_cols),
  ('ord', ordinal_prep, ordinal_cols)
])

In [ ]:
# checa o balanceamento das classes
def show_imbalance_ratio(df, name):
  df[target_col[0]].value_counts(normalize=True).plot(kind='bar')
  plt.title(f'Distribuição do Target - {name}')
  plt.xticks(rotation=0)
  plt.ylim(0, 1.0)
  plt.show()

show_imbalance_ratio(df_train, 'df_train')
show_imbalance_ratio(df_test1, 'df_test1')
show_imbalance_ratio(df_test2, 'df_test2')

In [ ]:
# pipeline
pipeline_prep = Pipeline(
  steps = [
    ('age_format', age_formatter),
    ('age_cat', age_categorizer),
    ('selector', column_selector),
    ('mapper', column_mapper),
    ('pre', preprocessor)
  ]
)

In [ ]:
# construção da rede neural
from tensorflow import keras
from tensorflow.keras import layers, regularizers

X_train_prepared = pipeline_prep.fit_transform(X_train, y_train).astype('float32')

inputs = layers.Input(shape=(X_train_prepared.shape[1],))

# camada 1: 2048 neurônios
x1 = layers.Dense(2048, kernel_regularizer=regularizers.l2(1e-4))(inputs)
x1 = layers.Activation('swish')(x1)
x1 = layers.BatchNormalization()(x1)
x1 = layers.Dropout(0.4)(x1)

# camada 2: 1024 neurônios
x2 = layers.Dense(1024, kernel_regularizer=regularizers.l2(1e-4))(x1)
x2 = layers.Activation('swish')(x2)
x2 = layers.BatchNormalization()(x2)
x2 = layers.Dropout(0.4)(x2)

atalho_1 = layers.Dense(1024, use_bias=False)(x1)
x2 = layers.Add()([x2, atalho_1])

# camada 3: 512 neurônios
x3 = layers.Dense(512, kernel_regularizer=regularizers.l2(1e-4))(x2)
x3 = layers.Activation('swish')(x3)
x3 = layers.BatchNormalization()(x3)
x3 = layers.Dropout(0.3)(x3)

# camada 4: 256 neurônios
x4 = layers.Dense(256, kernel_regularizer=regularizers.l2(1e-5))(x3)
x4 = layers.Activation('swish')(x4)
x4 = layers.BatchNormalization()(x4)
x4 = layers.Dropout(0.2)(x4)

# conectando camada 2 direto na saída da camada 4
atalho_2 = layers.Dense(256, use_bias=False)(x2)
x4 = layers.Add()([x4, atalho_2])

# camada 5: 128 neurônios
x5 = layers.Dense(128)(x4)
x5 = layers.Activation('swish')(x5)
x5 = layers.BatchNormalization()(x5)
x5 = layers.Dropout(0.1)(x5)

# camada 6: 64 neurônios
x6 = layers.Dense(64)(x5)
x6 = layers.Activation('swish')(x6)
x6 = layers.BatchNormalization()(x6)

# conectando camada 4 direto na saída da camada 6
atalho_3 = layers.Dense(64, use_bias=False)(x4)
x6 = layers.Add()([x6, atalho_3])

outputs = layers.Dense(1, activation='sigmoid')(x6)

model = keras.Model(inputs=inputs, outputs=outputs)

lr_schedule = keras.optimizers.schedules.CosineDecayRestarts(
  initial_learning_rate=0.001,
  first_decay_steps=1000,
  t_mul=2.0,
  m_mul=0.9
)

# compilação
model.compile(
  optimizer=keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=0.005),
  loss=keras.losses.BinaryFocalCrossentropy(gamma=2.0, alpha=0.25), 
  metrics=[
    keras.metrics.BinaryAccuracy(name='accuracy'),
    keras.metrics.Recall(name='recall'),
    keras.metrics.Precision(name='precision'),
    keras.metrics.F1Score(name='f1_score', threshold=0.5),
    keras.metrics.AUC(name='auc')
  ]
)

In [ ]:
# callbacks
early_stop = keras.callbacks.EarlyStopping(
  monitor='val_f1_score',
  patience=20,
  restore_best_weights=True,
  mode='max'
)

In [ ]:
# cria pesos para as classes do df_train
from sklearn.utils import class_weight

def calc_weights(y_calc):
  weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_calc), y=y_calc)
  weights_dict = dict(enumerate(weights))
  return weights_dict

weight_dict_train = calc_weights(y_train)
print(weight_dict_train)

In [ ]:
# separa e prepara dados do df_test1
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score,roc_auc_score, RocCurveDisplay

X_test1 = df_test1.drop(columns=target_col)
y_test1 = df_test1[target_col[0]]

X_test1_prepared = pipeline_prep.transform(X_test1).astype('float32')
y_test1 = y_test1.astype('float32')

In [ ]:
# treinamento inicial
history_initial = model.fit(
  X_train_prepared, y_train,
  validation_data=(X_test1_prepared, y_test1),
  epochs=100,
  batch_size=252,
  class_weight=weight_dict_train,
  callbacks=[early_stop],
  verbose=1
)

In [ ]:
# métricas de validação test1
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score

threshold_test1 = 0.5
y_proba_test1 = model.predict(X_test1_prepared).flatten()
y_pred_test1 = (y_proba_test1 >= threshold_test1).astype(int)

def calc_scores(y_test, y_pred, y_proba):
  scores = {
    "accuracy":[accuracy_score(y_test, y_pred)],
    "recall":[recall_score(y_test, y_pred)],
    "precision":[precision_score(y_test, y_pred)],
    "f1_score":[f1_score(y_test, y_pred)],
    "auc_roc":[roc_auc_score(y_test, y_proba)]
  }
  return scores

scores_test1 = calc_scores(y_test1, y_pred_test1, y_proba_test1)
pd.DataFrame(scores_test1, index=["Test1"])

In [ ]:
# curva ROC test1
from sklearn.metrics import RocCurveDisplay

def show_graphic_auc_roc(y_test, y_proba):
  RocCurveDisplay.from_predictions(y_test, y_proba, plot_chance_level=True)

show_graphic_auc_roc(y_test1, y_proba_test1)

In [ ]:
# curvas de aprendizagem test1
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].plot(history_initial.history['loss'], label='treino')
axes[0].plot(history_initial.history['val_loss'], label='validação')
axes[0].set_title('Loss')
axes[0].legend()

axes[1].plot(history_initial.history['auc'], label='treino')
axes[1].plot(history_initial.history['val_auc'], label='validação')
axes[1].set_title('AUC')
axes[1].legend()

axes[2].plot(history_initial.history['f1_score'], label='treino')
axes[2].plot(history_initial.history['val_f1_score'], label='validação')
axes[2].set_title('F1_Score')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# separa e prepara dados do df_test2
X_test2 = df_test2.drop(columns=target_col)
y_test2 = df_test2[target_col[0]]

X_test2_prepared = pipeline_prep.transform(X_test2).astype('float32')
y_test2 = y_test2.astype('float32')

In [ ]:
# cria pesos para as classes do df_test1
weights_dict_test1 = calc_weights(y_test1)
print(weights_dict_test1)

In [ ]:
# treinamento incremental
history_incremental = model.fit(
  X_test1_prepared, y_test1,
  validation_data=(X_test2_prepared, y_test2),
  epochs=100,
  batch_size=252,
  class_weight=weights_dict_test1,
  callbacks=[early_stop],
  verbose=1
)

In [ ]:
# métricas de validação test2
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score

threshold_test2 = 0.5
y_proba_test2 = model.predict(X_test2_prepared).flatten()
y_pred_test2 = (y_proba_test2 >= threshold_test2).astype(int)

scores_test2 = calc_scores(y_test2, y_pred_test2, y_proba_test2)
pd.DataFrame(scores_test2, index=["Test2"])

In [ ]:
# curva ROC test2
show_graphic_auc_roc(y_test2, y_proba_test2)

In [ ]:
# curvas de aprendizagem test2
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].plot(history_incremental.history['loss'], label='treino')
axes[0].plot(history_incremental.history['val_loss'], label='validação')
axes[0].set_title('Loss')
axes[0].legend()

axes[1].plot(history_incremental.history['auc'], label='treino')
axes[1].plot(history_incremental.history['val_auc'], label='validação')
axes[1].set_title('AUC')
axes[1].legend()

axes[2].plot(history_incremental.history['f1_score'], label='treino')
axes[2].plot(history_incremental.history['val_f1_score'], label='validação')
axes[2].set_title('F1_Score')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# wrapper para o modelo
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score

class KerasWrapper:
  def __init__(self, model):
    self.model = model
    self._estimator_type = "classifier"

  def fit(self, X, y):
    return self

  def predict(self, X):
    return self.model.predict(X, verbose=0)

  def score(self, X, y):
    preds = self.predict(X)
    return roc_auc_score(y, preds.ravel())

wrapper = KerasWrapper(model)

# seleciona amostra
sample_size = min(2000, len(X_test1_prepared))
index = np.random.choice(len(X_test1_prepared), sample_size, replace=False)
X_sample = X_test1_prepared[index]
y_sample = y_test1[index]

result = permutation_importance(wrapper, X_sample, y_sample, n_repeats=1, random_state=42, scoring=None)
lvl_importance = result.importances_mean

In [ ]:
# extração e ordenação das feature importances
final_cols = pipeline_prep.named_steps['pre'].get_feature_names_out()
df_importance = pd.DataFrame({
  'feature': final_cols,
  'importance': lvl_importance
})
df_importance = df_importance.sort_values('importance', ascending=False).reset_index(drop=True)

display(df_importance.head(30))

In [ ]:
# top features
top_features = df_importance.head(30)

plt.figure(figsize=(10, 12))
plt.barh(top_features['feature'][::-1], top_features['importance'][::-1])
plt.title('Top Features')
plt.xlabel('Queda no AUC')
plt.tight_layout()
plt.show()

In [ ]:
# matriz confusão
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import seaborn as sns

auc_test2 = roc_auc_score(y_test2, y_proba_test2)
print(f"AUC no Teste: {auc_test2:.4f}")
print("\nClassification Report:")
print(classification_report(y_test2, y_pred_test2))

plt.figure(figsize=(6, 4))
confusion_matriz = confusion_matrix(y_test2, y_pred_test2)
sns.heatmap(confusion_matriz, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xlabel('Predito')
plt.ylabel('Real')
plt.title('Matriz de Confusão - Test2')
plt.show()

In [ ]:
# extrai o pipeline de preparação e o modelo de rede neural
import cloudpickle

open('rede_neural_pipeline_preprocessing.pkl', 'wb').write(cloudpickle.dumps(pipeline_prep))
model.save('Modelo_rede.keras')